# Projet : Stage
### Objectif

Pour les attributs spatiaux et temporels, on voudrait faire la même chose. Pour cette partie, il y a 3 étapes à faire : 

1. Identifier les attributs spatiaux / temporels. 
2. Trouver le niveau d'hiérarchie de chaque attributs selon les hiérarchies spatiales / temporelles
3. Identifier le niveau d'hiérarchie le plus fin parmis tous les attributs spatiaux / temporels en tant que granularité minimum de dataset; identifier l'attribut au niveau d'hiérarchie le plus haut parmis tous les attributs en tant que scope de dataset et donner la liste de ses valeurs distinctes. 

L'output final qu'on demande est un dossier json de métadonnée de tous les datasets.

## 1. Hiérarchisation des données

In [ ]:
import json
import csv

def construire_dictionnaire_hierarchise():

    def fillChamp(dicChamps, dic_hierarchise, rang):
        for i in range(len(dicChamps['features'])):
            champ = dicChamps['features'][i]['properties']['nom']
            if champ not in dic_hierarchise[rang]:
                dic_hierarchise[rang].append(champ.lower())
        return

    def fillDictionnaireGeoJSON():
        fichiers = ['arrondissements', 'cantons', 'communes', 'departements', 'regions']
        dic_hierarchise = {}

        for fichier in fichiers:
            dic_hierarchise[fichier] = []
            with open(f"Education/levels/france-geojson/{fichier}-avec-outre-mer.geojson", "r", encoding="utf-8") as mon_json:
                data = json.load(mon_json)
                fillChamp(data, dic_hierarchise, fichier)

        return dic_hierarchise

    def fillDictionnaireQuartiers(dic_hierarchise):
        with open("liste-correspondance-qp2024-qp2015.csv", "r", encoding="utf-8") as fichier:
            reader = csv.reader(fichier, delimiter=";")
            listeQuartiers = list(reader)[1:]  # Ignorer l'en-tête

        dic_hierarchise['quartiers'] = []
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][1]
            if quartier not in dic_hierarchise['quartiers'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())

        dic_hierarchise['QP'] = []
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][3]
            if quartier not in dic_hierarchise['QP'] and quartier != "":
                dic_hierarchise['QP'].append(quartier.lower())

    dic_hierarchise = fillDictionnaireGeoJSON()
    fillDictionnaireQuartiers(dic_hierarchise)    

    # Tri des listes dans le dictionnaire
    champs_ranges = ['regions', 'departements', 'communes', 'cantons', 'arrondissements', 'quartiers', 'QP']
    dic_hierarchise = {champ: dic_hierarchise[champ] for champ in champs_ranges if champ in dic_hierarchise}
    
    return dic_hierarchise

def recuperer_dictionnaire_hierarchise():
    try:
        with open("dic_hierarchise.json", "r", encoding="utf-8") as fichier:
            dic_hierarchise = json.load(fichier)
    except FileNotFoundError:
        dic_hierarchise = construire_dictionnaire_hierarchise()
        with open("dic_hierarchise.json", "w", encoding="utf-8") as fichier:
            json.dump(dic_hierarchise, fichier, ensure_ascii=False, indent=4)
    
    return dic_hierarchise

In [24]:
# Appel de la fonction pour obtenir le dictionnaire hiérarchisé
# dic_hierarchise = recuperer_dictionnaire_hierarchise()
champs_ranges = ['regions', 'departements', 'communes', 'cantons', 'arrondissements', 'quartiers', 'QP']
hierarchie_champs = {champ: (len(champs_ranges) - i) for i, champ in enumerate(champs_ranges)}

## 2. Identification des attributs spatiaux dans un fichier csv/xlsx

### 2.1 Récupérer tous les attributs spatiaux

#### 2.1.1 Recuperation de tous les datasets

In [3]:
import os

def getFiles(origine='Opendata'):
    fichiers = []

    def separateurFichier(datasets):
        datasetSepares = {}

        for dataset in datasets:
            if dataset.endswith('.csv'):
                if 'csv' not in datasetSepares:
                    datasetSepares['csv'] = []
                datasetSepares['csv'].append(dataset)

            elif dataset.endswith('.xlsx'):
                if 'xlsx' not in datasetSepares:
                    datasetSepares['xlsx'] = []
                datasetSepares['xlsx'].append(dataset)

        return datasetSepares

    for dossier in os.walk(origine):
        for fichier in dossier[2]:
            if fichier.endswith('.csv') or fichier.endswith('.xlsx'):
                fichiers.append(os.path.join(dossier[0], fichier))
    
    return separateurFichier(fichiers)

datasets = getFiles()

#### 2.1.2 Recherche des attributs spatiaux via contenu des cellules

Fonctions utiles

In [ ]:
def estSpatial(attribut, dic_hierarchise):
    for champ, valeurs in dic_hierarchise.items():
        if attribut in valeurs:
            return [True, champ]
    return [False, None]

def recupererAttributsSpatiaux(headers, df, dic_hierarchise, score_colonne):

    liste_attributs_spatiaux = {}
    for i in range(10):
        for j in range(len(headers)):
            cell = df.iloc[i, j]
            if isinstance(cell, str):
                cell = cell.lower()

            infoSpatial = estSpatial(cell, dic_hierarchise)
            if infoSpatial[0]:
                score_colonne[j] += 1
                if (score_colonne[j]*10) >= 50 and headers[j] not in liste_attributs_spatiaux.keys():
                    liste_attributs_spatiaux[headers[j]] = [cell, infoSpatial[1]]
    
    return liste_attributs_spatiaux

def recupererLowGranAndScope()

In [ ]:
import load_file as lf

compteur_datasets = 0

for dataset in datasets['csv']:
    nom_fichier = dataset.split('/')[-1]
    extension = nom_fichier.split('.')[-1]
    compteur_datasets += 1
    score_colonne = {}
    liste_attributs_spatiaux = {}
    low_gran_and_scope = {}

    print(f'Traitement du fichier : {dataset[:45]}... | {compteur_datasets}/{len(datasets['csv'])}')
    
    try:
        df = lf.find_type(dataset)[0]
        headers = df.columns.tolist()
        for k in range(len(headers)):
            score_colonne[k] = 0
        print(f"En-têtes du fichier {nom_fichier[:10]}...{extension}: {headers}")
    except Exception as e:
        print(f"Erreur lors du chargement du fichier {nom_fichier[:10]}: {e}")
        continue

    liste_attributs_spatiaux = recupererAttributsSpatiaux(headers, df, dic_hierarchise, score_colonne)
    
    if len(liste_attributs_spatiaux) > 0:
        le_plus_haut = ['QP', 1]
        le_plus_bas = ['regions', 7]

        for attribut, champ in liste_attributs_spatiaux.items():
            if hierarchie_champs[champ[1]] < hierarchie_champs[le_plus_bas[0]]:
                le_plus_bas = [champ[1], attribut]
            if hierarchie_champs[champ[1]] > hierarchie_champs[le_plus_haut[0]]:
                le_plus_haut = [champ[1], attribut]
    else:
        le_plus_haut = None
        le_plus_bas = None    
    
    low_gran_and_scope[nom_fichier] = {'Low granularity': le_plus_bas, 'Scope': le_plus_haut}

    print(low_gran_and_scope)
        
    # low_gran_and_scope[nom_fic] = {'Low granularity': le_plus_bas, 'Scope': le_plus_haut}

    if compteur_datasets == 5:
        break

Traitement du fichier : Opendata/Landes/Education/fr-en-adresse-et-ge... | 1/52
En-têtes du fichier fr-en-adre...csv: ['numero_uai', 'appellation_officielle', 'denomination_principale', 'patronyme_uai', 'secteur_public_prive_libe', 'adresse_uai', 'lieu_dit_uai', 'boite_postale_uai', 'code_postal_uai', 'localite_acheminement_uai', 'libelle_commune', 'coordonnee_x', 'coordonnee_y', 'EPSG', 'latitude', 'longitude', 'appariement', 'localisation', 'nature_uai', 'nature_uai_libe', 'etat_etablissement', 'etat_etablissement_libe', 'code_departement', 'code_region', 'code_academie', 'code_commune', 'libelle_departement', 'libelle_region', 'libelle_academie', 'position', 'secteur_prive_code_type_contrat', 'secteur_prive_libelle_type_contrat', 'code_ministere', 'libelle_ministere', 'date_ouverture']
{'fr-en-adresse-et-geolocalisation-etablissements-premier-et-second-degre.csv': {'Low granularity': ['communes', 'libelle_commune'], 'Scope': ['departements', 'libelle_departement']}}
